## What is a candidate panel, and why does it exist?

A **candidate panel** is the precomputed, persisted output of the expensive part of the pipeline — causal residualization + pair candidate construction — decoupled from simulation.

**Rationale:** residual computation (rolling/expanding regressions per stock) and candidate discovery (hedge ratios, cointegration checks, half-life/kappa filters) are the costly steps. Simulation — sizing, entry/exit logic, sweeping parameters like `entry_z` — is comparatively cheap and gets run *many times over* against the same candidates during research. A candidate panel is computed once per (sector, residual config) and reused across every downstream simulation that needs it.

**What it consists of:**
- **Panel** (`.panel.parquet`) — one row per `(pair, asof_date)`, with hedge ratios, spread diagnostics, and filter-pass metadata for every candidate pair discovered at that date.
- **`residual_params`** (`_residual_params.pkl`) — the fitted residualization parameters needed to reconstruct residual series without refitting.
- **`meta.json`** — the residual config used to produce the panel, plus date range and row counts — lets downstream code confirm it's loading the right panel for a given `residual_key`.

Stock/spread-level **`series/`** is a separate, optional persistence step — not part of the panel itself, and recomputed on demand if absent.

## How to create a candidate panel

This notebook builds one panel for the **Materials** sector, persists it, and loads it back.

The entry point is `run_panel_batch(PanelBatchConfig(...))` from `src.candidates.panel_batch` — the same function the `residuals` stage of `run_me.py` calls.

## Panel creation has three independent steps

1. **Fit residuals** — causal regression of each stock against market/sector, per `residual_mode`.
2. **Fit hedge ratios** — OLS or PCA on the fitted residuals, over a window of `hedge_ratio_lb` days. Unweighted.
3. **Compute MR diagnostics** — kappa, half-life, mr_score, over a window of `mr_diag_lb` days. Unweighted.

Steps 2 and 3 are independent of each other and of the residual-fitting axis — you can fit residuals on a long, stable window while scoring mean-reversion diagnostics on a shorter, more recent one, or vice versa.

## Residual modes — pick one

| `residual_mode` | weighting | window | swept knob | `min_history` |
| --- | --- | --- | --- | --- |
| `EQ_ROLLING` | equal | rolling | `residual_lb` | = `residual_lb` |
| `EQ_EXPANDING` | equal | expanding | *(none)* | `residual_min_lb_eq_exp` |
| `DECAY_EXPANDING` | exponential decay | expanding | `residual_hl` | `residual_hl * residual_min_lb_dec_exp` (or absolute, see `residual_min_lb_type_dec_exp`) |

`decay_rolling` doesn't exist — time-decay + rolling window is redundant with `DECAY_EXPANDING` and deliberately excluded.

## Knobs

Set `RESIDUAL_MODE` first — it determines which of the other knobs are active. Only the knobs relevant to the chosen mode are used; the rest are ignored (see the `match` block below).

In [1]:
from src.candidates.panel_batch import PanelBatchConfig, run_panel_batch
from src.simulator.config import ResidualMode, AbsOrMult

# ── KNOB: pick the residual mode ─────────────────────────────────────────
RESIDUAL_MODE = ResidualMode.DECAY_EXPANDING   # try: EQ_ROLLING, EQ_EXPANDING

# ── KNOB: DECAY_EXPANDING params (ignored unless RESIDUAL_MODE == DECAY_EXPANDING) ──
RESIDUAL_HL = 504                              # -> exp_hl504
MIN_HIST_TYPE_DEC_EXP = AbsOrMult.MULTIPLIER   # MULTIPLIER or ABSOLUTE
MIN_HIST_VALUE_DEC_EXP = 2                     # multiplier: min_history = hl*2 = 1008 -> mh1008
                                                # absolute:   min_history = this value directly

# ── KNOB: EQ_ROLLING params (ignored unless RESIDUAL_MODE == EQ_ROLLING) ──
RESIDUAL_LB = 126                              # -> rol_lb126; min_history = residual_lb, always

# ── KNOB: EQ_EXPANDING params (ignored unless RESIDUAL_MODE == EQ_EXPANDING) ──
MIN_HIST_EQ_EXP = 252                          # -> exp_mh252; no swept axis in this mode

# ── KNOB: shared across all modes — steps 2 and 3, independent of each other ──
HEDGE_RATIO_LB = 252                           # window for OLS/PCA hedge-ratio fit
MR_DIAG_LB = 252                               # window for mean-reversion diagnostics
                                                # try MR_DIAG_LB=63 or 126 with HEDGE_RATIO_LB=252
                                                # held fixed, to test shorter-window diagnostics

# ── KNOB: risk-free rate subtraction ─────────────────────────────────────
SUBTRACT_RISK_FREE = True                      # -> _rf suffix on residual_key

# ── KNOB: sector / universe; None=all available sectors  ───────────────────────────────────────────────
SELECTED_SECTORS = ["materials"]

# ── KNOB: notebook speed cap — omit / set None for full history ──────────
MAX_STEPS = 8

In [2]:
# Build the mode-specific fields — only the ones relevant to RESIDUAL_MODE are set.
match RESIDUAL_MODE:
    case ResidualMode.DECAY_EXPANDING:
        mode_kwargs = dict(
            residual_hl=RESIDUAL_HL,
            residual_min_lb_type_dec_exp=MIN_HIST_TYPE_DEC_EXP,
            residual_min_lb_dec_exp=MIN_HIST_VALUE_DEC_EXP,
        )
    case ResidualMode.EQ_ROLLING:
        mode_kwargs = dict(residual_lb=RESIDUAL_LB)
    case ResidualMode.EQ_EXPANDING:
        mode_kwargs = dict(residual_min_lb_eq_exp=MIN_HIST_EQ_EXP)

cfg = PanelBatchConfig(
    residual_mode=RESIDUAL_MODE,
    hedge_ratio_lb=HEDGE_RATIO_LB,
    mr_diag_lb=MR_DIAG_LB,
    subtract_risk_free=SUBTRACT_RISK_FREE,
    selected_sectors=SELECTED_SECTORS,
    frequency="W-FRI",
    max_steps=MAX_STEPS,               # notebook-only cap; omit for full history
    persist_result=True,
    persist_residual_params=True,
    persist_dir_template="howto",      # subdir under settings.CANDIDATE_PANELS_ROOT
    **mode_kwargs,
)

# resolved_universe_dir()/resolved_data_path() fall back to settings when unset.
results = run_panel_batch(cfg)

Batch 20260721_1311
Sectors: 10 [dsc, stp, nrg, fin, hlt, ind, tec, mat, rle, utl]
Residual mode: decay_expanding
Sweep points: [504] (['exp_hl504_mh1008_rf'])
Total jobs: 10


Sectors:   0%|          | 0/10 [00:00<?, ?sector/s]

Loading universe consumer_discretionary_only_v1...
  dsc / decay_expanding/504: 2370/2400 valid candidates
Loading universe consumer_staples_only_v1...
  stp / decay_expanding/504: 2275/2275 valid candidates
Loading universe energy_only_v1...
  nrg / decay_expanding/504: 439/440 valid candidates
Loading universe financials_only_v1...
  fin / decay_expanding/504: 2093/2100 valid candidates
Loading universe health_care_only_v1...
  hlt / decay_expanding/504: 2299/2400 valid candidates
Loading universe industrials_only_v1...
  ind / decay_expanding/504: 2398/2400 valid candidates
Loading universe information_technology_only_v1...
  tec / decay_expanding/504: 2048/2100 valid candidates
Loading universe materials_only_v1...
  mat / decay_expanding/504: 624/624 valid candidates
Loading universe real_estate_only_v1...
  rle / decay_expanding/504: 793/840 valid candidates
Loading universe utilities_only_v1...
  utl / decay_expanding/504: 2400/2400 valid candidates

Done. 10 panels created.


`run_panel_batch` returns a dict keyed by `(group_id, residual_mode, swept_value)`. Each value is a `CandidatePanelResult` with a `.panel` DataFrame and a `.metadata` dict.

In [3]:
for key, result in results.items():
    print(f"key: {key}")
    print(f"rows: {len(result.panel)}  valid: {int(result.panel['is_valid'].sum())}")
    print(f"residual key: {result.metadata['residual_cfg']}")

next(iter(results.values())).panel.head()

key: ('consumer_discretionary', 'decay_expanding', 504)
rows: 2400  valid: 2370
residual key: {'window_mode': 'expanding', 'lookback': 252, 'half_life': 504, 'min_history': 1008, 'remove_residual_pcs': 0, 'subtract_risk_free': True}
key: ('consumer_staples', 'decay_expanding', 504)
rows: 2275  valid: 2275
residual key: {'window_mode': 'expanding', 'lookback': 252, 'half_life': 504, 'min_history': 1008, 'remove_residual_pcs': 0, 'subtract_risk_free': True}
key: ('energy', 'decay_expanding', 504)
rows: 440  valid: 439
residual key: {'window_mode': 'expanding', 'lookback': 252, 'half_life': 504, 'min_history': 1008, 'remove_residual_pcs': 0, 'subtract_risk_free': True}
key: ('financials', 'decay_expanding', 504)
rows: 2100  valid: 2093
residual key: {'window_mode': 'expanding', 'lookback': 252, 'half_life': 504, 'min_history': 1008, 'remove_residual_pcs': 0, 'subtract_risk_free': True}
key: ('health_care', 'decay_expanding', 504)
rows: 2400  valid: 2299
residual key: {'window_mode': 'expa

,candidate_type,candidate_subtype,weight_model,spread_id,group_id,n_legs,weights,adf_pvalue,mr_score,kappa,half_life,residual_std,spread_return_std,level_std,is_valid,candidate_id,asof_datetime,asof_date
0,pair,pca,pca,AMZN|AZO,consumer_discretionary,2,"{""AMZN"":1.0,""AZO"":-8.476956263205}",0.597654,0.090160,0.015508,44.695721,0.172007,0.172471,1.075001,True,84029478d2dd5724,2009-07-24,2009-07-24
1,pair,pca,pca,AMZN|BBY,consumer_discretionary,2,"{""AMZN"":1.0,""BBY"":-3.62756043093}",0.400902,0.270415,0.023349,29.686378,0.086345,0.086719,0.411449,True,7568d3cce9102bd1,2009-07-24,2009-07-24
2,pair,pca,pca,AMZN|BKNG,consumer_discretionary,2,"{""AMZN"":1.0,""BKNG"":-0.572027473461}",0.148411,1.487408,0.044069,15.728622,0.029628,0.029903,0.101106,True,53a9738d9fbe049e,2009-07-24,2009-07-24
3,pair,pca,pca,AMZN|DHI,consumer_discretionary,2,"{""AMZN"":1.0,""DHI"":-0.034027207317}",0.451008,0.777422,0.021697,31.947024,0.027909,0.028008,0.135176,True,2b83b883717fdc10,2009-07-24,2009-07-24
4,pair,pca,pca,AMZN|DRI,consumer_discretionary,2,"{""AMZN"":1.0,""DRI"":-0.642673920726}",0.029007,2.158282,0.069051,10.038223,0.031993,0.032533,0.089907,True,2fafb4e71019074b,2009-07-24,2009-07-24
